In [1]:
!pip install polars --quiet
import polars as pl
import numpy as np
import os
import gc

print("Polars version:", pl.__version__)

Polars version: 1.35.2


In [2]:
dataset_path = "/content/drive/MyDrive/Colab Notebooks"  # Sesuaikan dengan path Anda

csv_files = [
    "Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-16-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-23-02-2018_TrafficForML_CICFlowMeter.csv",
    "Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv",
    "Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv",
    "Friday-02-03-2018_TrafficForML_CICFlowMeter.csv"
]

print(f"Total file: {len(csv_files)}")
for f in csv_files:
    print(" -", f)

Total file: 10
 - Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv
 - Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv
 - Friday-16-02-2018_TrafficForML_CICFlowMeter.csv
 - Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv
 - Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv
 - Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv
 - Friday-23-02-2018_TrafficForML_CICFlowMeter.csv
 - Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv
 - Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv
 - Friday-02-03-2018_TrafficForML_CICFlowMeter.csv


In [3]:
# Dictionary untuk menyimpan nama kolom per file
columns_per_file = {}

for f in csv_files:
    path = os.path.join(dataset_path, f)
    # Scan CSV, baca hanya baris pertama (header) dengan n_rows=0
    lf = pl.scan_csv(path, n_rows=0)
    cols = lf.collect_schema().names()
    # Bersihkan spasi di nama kolom
    cols = [col.strip() for col in cols]
    columns_per_file[f] = cols
    print(f"{f}: {len(cols)} kolom")
    # Tampilkan 10 kolom pertama
    print("   Contoh:", cols[:10])

Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
   Contoh: ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min']
Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
   Contoh: ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min']
Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
   Contoh: ['Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Fwd Pkt Len Max', 'Fwd Pkt Len Min']
Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv: 84 kolom
   Contoh: ['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp', 'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts']
Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 80 kolom
   Contoh: ['Dst Port', 'Protocol', '

In [4]:
common_cols = set(columns_per_file[csv_files[0]])
for f in csv_files[1:]:
    common_cols = common_cols.intersection(columns_per_file[f])

common_cols = sorted(list(common_cols))
print(f"Jumlah kolom yang umum di semua file: {len(common_cols)}")
print("Contoh kolom umum:", common_cols[:10])

Jumlah kolom yang umum di semua file: 80
Contoh kolom umum: ['ACK Flag Cnt', 'Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Bwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Header Len', 'Bwd IAT Max', 'Bwd IAT Mean']


In [5]:
# Kolom identitas yang harus dihapus
identity_cols = ['Dst Port', 'Protocol', 'Timestamp', 'Flow ID', 'Src IP', 'Dst IP', 'Src Port']
# Hanya yang ada di common_cols
to_remove = [col for col in identity_cols if col in common_cols]

# Kolom fitur yang akan dipertahankan
keep_cols = [col for col in common_cols if col not in to_remove]

# Pastikan Label ada
if 'Label' in common_cols:
    keep_cols.append('Label')
else:
    print("Peringatan: Label tidak ditemukan di common_cols. Akan ditambahkan secara manual.")
    keep_cols.append('Label')

# Hapus duplikat
keep_cols = list(set(keep_cols))
print(f"Jumlah kolom yang akan dipertahankan (termasuk Label): {len(keep_cols)}")
print("Contoh kolom:", sorted(keep_cols)[:10])
print("Apakah Label ada?", 'Label' in keep_cols)

Jumlah kolom yang akan dipertahankan (termasuk Label): 77
Contoh kolom: ['ACK Flag Cnt', 'Active Max', 'Active Mean', 'Active Min', 'Active Std', 'Bwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Header Len', 'Bwd IAT Max', 'Bwd IAT Mean']
Apakah Label ada? True


In [6]:
lazy_frames = []

for f in csv_files:
    path = os.path.join(dataset_path, f)
    print(f"Memproses {f}...")
    # Baca sebagai string (infer_schema_length=0)
    lf = pl.scan_csv(path, infer_schema_length=0, ignore_errors=False)
    # Bersihkan spasi pada nama kolom
    rename_map = {col: col.strip() for col in lf.collect_schema().names()}
    lf = lf.rename(rename_map)
    # Pilih kolom yang ada di keep_cols
    available = [col for col in keep_cols if col in lf.collect_schema().names()]
    lf = lf.select(available)
    # Hapus baris header ganda (jika ada)
    if 'Label' in available:
        lf = lf.filter(pl.col("Label") != "Label")
        # Koreksi typo label
        lf = lf.with_columns(pl.col("Label").str.replace("Infilteration", "Infiltration"))
    lazy_frames.append(lf)
    gc.collect()

print("Semua lazy frame siap.")

Memproses Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv...
Memproses Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv...
Memproses Friday-16-02-2018_TrafficForML_CICFlowMeter.csv...
Memproses Thuesday-20-02-2018_TrafficForML_CICFlowMeter.csv...
Memproses Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv...
Memproses Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv...
Memproses Friday-23-02-2018_TrafficForML_CICFlowMeter.csv...
Memproses Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv...
Memproses Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv...
Memproses Friday-02-03-2018_TrafficForML_CICFlowMeter.csv...
Semua lazy frame siap.


In [7]:
non_label_cols = [col for col in keep_cols if col != 'Label']

casted_frames = []
for lf in lazy_frames:
    lf = lf.with_columns([
        pl.col(col).cast(pl.Float32, strict=False)
        for col in non_label_cols if col in lf.collect_schema().names()
    ])
    casted_frames.append(lf)

print("Casting selesai.")

Casting selesai.


In [8]:
print("Menggabungkan semua lazy frame...")
full_lf = pl.concat(casted_frames)

print("Mengumpulkan data (collect dengan streaming)...")
final_df = full_lf.collect(engine="streaming")

print(f"Shape data setelah collect: {final_df.shape}")
print(f"Perkiraan memori: {final_df.estimated_size() / (1024**2):.2f} MB")

Menggabungkan semua lazy frame...
Mengumpulkan data (collect dengan streaming)...
Shape data setelah collect: (16232943, 77)
Perkiraan memori: 4825.03 MB


In [9]:
label_counts = final_df.group_by('Label').agg(pl.len())
print("Distribusi Label (belum diubah jadi biner):")
print(label_counts)

Distribusi Label (belum diubah jadi biner):
shape: (15, 2)
┌──────────────────────────┬────────┐
│ Label                    ┆ len    │
│ ---                      ┆ ---    │
│ str                      ┆ u32    │
╞══════════════════════════╪════════╡
│ DDOS attack-LOIC-UDP     ┆ 1730   │
│ DDOS attack-HOIC         ┆ 686012 │
│ SQL Injection            ┆ 87     │
│ DDoS attacks-LOIC-HTTP   ┆ 576191 │
│ DoS attacks-SlowHTTPTest ┆ 139890 │
│ …                        ┆ …      │
│ SSH-Bruteforce           ┆ 187589 │
│ Infiltration             ┆ 161934 │
│ Brute Force -Web         ┆ 611    │
│ FTP-BruteForce           ┆ 193360 │
│ DoS attacks-Hulk         ┆ 461912 │
└──────────────────────────┴────────┘


In [10]:
null_counts = final_df.null_count()
null_summary = null_counts.to_pandas().T
null_summary.columns = ['null_count']
null_summary = null_summary[null_summary['null_count'] > 0]
null_summary = null_summary.sort_values('null_count', ascending=False)
print("Kolom dengan nilai null:")
print(null_summary.head(10))

Kolom dengan nilai null:
Empty DataFrame
Columns: [null_count]
Index: []


In [11]:
float_cols = [col for col in final_df.columns if final_df[col].dtype == pl.Float32]
inf_counts = {}
for col in float_cols:
    inf_cnt = final_df.filter(pl.col(col).is_infinite()).height
    if inf_cnt > 0:
        inf_counts[col] = inf_cnt
if inf_counts:
    print("Kolom dengan nilai tak hingga:")
    for col, cnt in inf_counts.items():
        print(f"  {col}: {cnt} baris")
else:
    print("Tidak ada nilai tak hingga.")

Kolom dengan nilai tak hingga:
  Flow Byts/s: 36039 baris
  Flow Pkts/s: 95760 baris


In [12]:
# Ganti inf dengan null
for col in float_cols:
    final_df = final_df.with_columns(
        pl.when(pl.col(col).is_infinite())
        .then(None)
        .otherwise(pl.col(col))
        .alias(col)
    )

# Hapus baris dengan null (termasuk yang berasal dari inf)
initial_rows = final_df.height
final_df = final_df.drop_nulls()
rows_after = final_df.height
print(f"Baris sebelum drop nulls: {initial_rows:,}")
print(f"Baris setelah drop nulls: {rows_after:,}")
print(f"Baris dihapus: {initial_rows - rows_after:,} ({100*(initial_rows-rows_after)/initial_rows:.2f}%)")

Baris sebelum drop nulls: 16,232,943
Baris setelah drop nulls: 16,137,183
Baris dihapus: 95,760 (0.59%)


In [13]:
final_df = final_df.with_columns([
    pl.when(pl.col("Init Fwd Win Byts") == -1).then(0).otherwise(pl.col("Init Fwd Win Byts")).alias("Init Fwd Win Byts"),
    pl.when(pl.col("Init Bwd Win Byts") == -1).then(0).otherwise(pl.col("Init Bwd Win Byts")).alias("Init Bwd Win Byts")
])

In [14]:
if 'Label' in final_df.columns:
    final_df = final_df.with_columns(
        pl.when(pl.col("Label").str.to_lowercase() == "benign")
        .then(0)
        .otherwise(1)
        .alias("attack")
    )
    final_df = final_df.drop("Label")
else:
    raise ValueError("Kolom Label tidak ditemukan!")

attack_counts = final_df.group_by('attack').agg(pl.len())
print("Distribusi attack (0=Benign, 1=Attack):")
print(attack_counts)

Distribusi attack (0=Benign, 1=Attack):
shape: (2, 2)
┌────────┬──────────┐
│ attack ┆ len      │
│ ---    ┆ ---      │
│ i32    ┆ u32      │
╞════════╪══════════╡
│ 0      ┆ 13390249 │
│ 1      ┆ 2746934  │
└────────┴──────────┘


In [15]:
num_cols = [col for col in final_df.columns if final_df[col].dtype == pl.Float32]
desc = final_df.select(num_cols).describe()
print(desc)

shape: (9, 77)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ statistic ┆ Bwd       ┆ Pkt Len   ┆ Bwd Pkt   ┆ … ┆ CWE Flag  ┆ Fwd Seg   ┆ Bwd       ┆ Pkt Size │
│ ---       ┆ Header    ┆ Mean      ┆ Len Std   ┆   ┆ Count     ┆ Size Min  ┆ Pkts/s    ┆ Avg      │
│ str       ┆ Len       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│           ┆ ---       ┆ f64       ┆ f64       ┆   ┆ f64       ┆ f64       ┆ f64       ┆ f64      │
│           ┆ f64       ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ count     ┆ 1.6137183 ┆ 1.6137183 ┆ 1.6137183 ┆ … ┆ 1.6137183 ┆ 1.6137183 ┆ 1.6137183 ┆ 1.613718 │
│           ┆ e7        ┆ e7        ┆ e7        ┆   ┆ e7        ┆ e7        ┆ e7        ┆ 3e7      │
│ null_coun ┆ 0.0       ┆ 0.0       ┆ 0.0       ┆ … ┆ 0.0       ┆ 0.0       

In [16]:
# Hitung varians setiap kolom numerik
num_cols = [col for col in final_df.columns if final_df[col].dtype == pl.Float32]
variance = final_df.select([pl.col(col).var() for col in num_cols])

# Tampilkan kolom dengan varians 0
zero_var_cols = []
for col in num_cols:
    var_val = variance[col][0]
    if var_val == 0:
        zero_var_cols.append(col)

print("Kolom dengan varians 0 (semua nilainya sama):")
print(zero_var_cols)

# Hapus kolom tersebut
if zero_var_cols:
    final_df = final_df.drop(zero_var_cols)
    print(f"{len(zero_var_cols)} kolom dihapus.")
else:
    print("Tidak ada kolom konstan.")

Kolom dengan varians 0 (semua nilainya sama):
['Bwd PSH Flags', 'Fwd Blk Rate Avg', 'Bwd URG Flags', 'Bwd Blk Rate Avg', 'Fwd Pkts/b Avg', 'Fwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Byts/b Avg']
8 kolom dihapus.


In [17]:
# Daftar kolom yang secara logika tidak boleh negatif
nonneg_cols = [
    'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts',
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Tot', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
    'Bwd IAT Tot', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
    'Active Mean', 'Active Std', 'Active Max', 'Active Min',
    'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min',
    'Pkt Len Min', 'Pkt Len Max', 'Pkt Len Mean', 'Pkt Len Std', 'Pkt Len Var'
]
# Ambil hanya kolom yang masih ada di final_df
nonneg_cols = [col for col in nonneg_cols if col in final_df.columns]

# Hitung baris dengan nilai negatif per kolom
neg_rows = {}
for col in nonneg_cols:
    cnt = final_df.filter(pl.col(col) < 0).height
    if cnt > 0:
        neg_rows[col] = cnt

if neg_rows:
    print("Kolom dengan nilai negatif:")
    for col, cnt in neg_rows.items():
        print(f"  {col}: {cnt} baris ({cnt/final_df.height*100:.4f}%)")
else:
    print("Tidak ada nilai negatif pada kolom non-negatif.")

Kolom dengan nilai negatif:
  Flow Duration: 14 baris (0.0001%)
  Flow IAT Mean: 14 baris (0.0001%)
  Flow IAT Max: 3 baris (0.0000%)
  Flow IAT Min: 15 baris (0.0001%)
  Fwd IAT Tot: 14 baris (0.0001%)
  Fwd IAT Mean: 14 baris (0.0001%)
  Fwd IAT Max: 3 baris (0.0000%)
  Fwd IAT Min: 15 baris (0.0001%)


In [18]:
if neg_rows:
    mask = None
    for col in neg_rows.keys():
        if mask is None:
            mask = (pl.col(col) >= 0)
        else:
            mask = mask & (pl.col(col) >= 0)
    final_df = final_df.filter(mask)
    print(f"Baris setelah menghapus nilai negatif: {final_df.height:,}")
else:
    print("Tidak ada baris dengan nilai negatif.")

Baris setelah menghapus nilai negatif: 16,137,168


In [19]:
output_path = os.path.join(dataset_path, "cicids2018_cleaned.parquet")
final_df.write_parquet(output_path, compression="snappy")
print(f"Data disimpan ke: {output_path}")
print(f"Shape akhir: {final_df.shape}")

Data disimpan ke: /content/drive/MyDrive/Colab Notebooks/cicids2018_cleaned.parquet
Shape akhir: (16137168, 69)


In [20]:
# Load data bersih
df = pl.read_parquet(os.path.join(dataset_path, "cicids2018_cleaned.parquet"))
print(f"Shape: {df.shape}")

Shape: (16137168, 69)


In [21]:
#initial_rows = df.height
#df = df.unique()
#print(f"Duplicates removed: {initial_rows - df.height} rows")
#print(f"New shape: {df.shape}")

In [22]:
skewed_cols = [
    'Flow Duration', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts',
    'Flow IAT Max', 'Flow IAT Mean', 'Fwd IAT Tot',
    'Bwd IAT Tot', 'Active Mean', 'Idle Mean'
]

skewed_present = [col for col in skewed_cols if col in df.columns]
df = df.with_columns([
    pl.col(col).log1p() for col in skewed_present
])

print(f"Log1p applied to {len(skewed_present)} features.")

Log1p applied to 9 features.


In [23]:
# Setelah df.unique()
num_cols = [col for col in df.columns if df[col].dtype == pl.Float32]
variance = df.select([pl.col(col).var() for col in num_cols])
zero_var_cols = [col for col in num_cols if variance[col][0] == 0]
if zero_var_cols:
    df = df.drop(zero_var_cols)
    print(f"Dropped constant columns after unique: {zero_var_cols}")

In [24]:
num_cols = [col for col in df.columns if df[col].dtype == pl.Float32]

df_pd = df.select(num_cols).to_pandas()

# 3. Temukan kolom identik
identical_cols_dict = {}
to_drop_identical = []

cols = df_pd.columns
for i in range(len(cols)):
    col1 = cols[i]
    if col1 in to_drop_identical:
        continue

    for j in range(i + 1, len(cols)):
        col2 = cols[j]
        if col2 in to_drop_identical:
            continue

        # Cek apakah isi kolom 100% sama
        if df_pd[col1].equals(df_pd[col2]):
            if col1 not in identical_cols_dict:
                identical_cols_dict[col1] = []
            identical_cols_dict[col1].append(col2)
            to_drop_identical.append(col2)

# 4. Eksekusi Penghapusan
if to_drop_identical:
    print("Identical columns found:")
    for main_col, dups in identical_cols_dict.items():
        print(f"  - {main_col} identik dengan {dups}")

    # Hapus kolom dari Polars DataFrame
    to_drop_identical = list(set(to_drop_identical))
    print(f"\nDropping {len(to_drop_identical)} columns: {to_drop_identical}")

    df = df.drop(to_drop_identical)

    # Update daftar num_cols agar tahap selanjutnya (korelasi) tidak error
    num_cols = [col for col in num_cols if col not in to_drop_identical]
    print(f"New shape after dropping identical: {df.shape}")
else:
    print("No identical columns found.")

# 5. Bersihkan memory Pandas setelah selesai (Opsional tapi baik dilakukan)
import gc
del df_pd
gc.collect()

Identical columns found:
  - Fwd Seg Size Avg identik dengan ['Fwd Pkt Len Mean']
  - SYN Flag Cnt identik dengan ['Fwd PSH Flags']
  - Subflow Fwd Pkts identik dengan ['Tot Fwd Pkts']
  - Subflow Bwd Pkts identik dengan ['Tot Bwd Pkts']
  - Fwd URG Flags identik dengan ['CWE Flag Count']
  - Bwd Pkt Len Mean identik dengan ['Bwd Seg Size Avg']

Dropping 6 columns: ['Tot Fwd Pkts', 'CWE Flag Count', 'Tot Bwd Pkts', 'Fwd Pkt Len Mean', 'Fwd PSH Flags', 'Bwd Seg Size Avg']
New shape after dropping identical: (16137168, 63)


0

In [25]:
from sklearn.model_selection import train_test_split
import joblib

# Konversi ke Pandas
X = df.drop('attack').to_pandas()
y = df['attack'].to_pandas()

del df, final_df
gc.collect()

# Split 70:15:15 (Stratified)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

Train: 11296017 | Val: 2420575 | Test: 2420576


In [26]:
print("[INFO] Menghitung Matriks Korelasi...")
corr_matrix_train = X_train.corr()
high_corr_pairs = np.where(np.abs(corr_matrix_train) > 0.95)
high_corr_pairs = [(corr_matrix_train.index[x], corr_matrix_train.columns[y], corr_matrix_train.iloc[x, y])
                   for x, y in zip(*high_corr_pairs) if x != y and x < y]

to_drop_corr = set()
if high_corr_pairs:
    for f1, f2, corr in high_corr_pairs:
        if f1 in to_drop_corr or f2 in to_drop_corr: continue
        avg_f1 = np.mean(np.abs(corr_matrix_train[f1].drop(f1)))
        avg_f2 = np.mean(np.abs(corr_matrix_train[f2].drop(f2)))
        if avg_f1 >= avg_f2: to_drop_corr.add(f1)
        else: to_drop_corr.add(f2)

    X_train = X_train.drop(columns=list(to_drop_corr))
    X_val   = X_val.drop(columns=list(to_drop_corr))
    X_test  = X_test.drop(columns=list(to_drop_corr))
    print(f"Berhasil membuang {len(to_drop_corr)} fitur redundan.")

with open(os.path.join(dataset_path, "dropped_features_correlation.txt"), "w") as f:
    for feat in sorted(list(to_drop_corr)):
        f.write(feat + "\n")

[INFO] Menghitung Matriks Korelasi...
Berhasil membuang 14 fitur redundan.


In [27]:
from sklearn.preprocessing import PowerTransformer, MinMaxScaler

print("\n--- STEP 2: STABILIZED YEO-JOHNSON PIPELINE ---")
# Tahap 1: MinMaxScaler awal untuk stabilitas
initial_scaler = MinMaxScaler()
X_train_init = initial_scaler.fit_transform(X_train)

# Tahap 2: Yeo-Johnson
pt = PowerTransformer(method='yeo-johnson', standardize=True)
sample_idx = np.random.choice(X_train_init.shape[0], min(100000, X_train_init.shape[0]), replace=False)
pt.fit(X_train_init[sample_idx])

# Tahap 3: Transformasi
X_train_pt = pt.transform(X_train_init)
del X_train_init
gc.collect()

X_val_pt  = pt.transform(initial_scaler.transform(X_val))
X_test_pt = pt.transform(initial_scaler.transform(X_test))

# Tahap 4: Final Scaling (0-1) untuk Autoencoder
final_scaler = MinMaxScaler()
X_train_scaled = final_scaler.fit_transform(X_train_pt)
X_val_scaled   = final_scaler.transform(X_val_pt)
X_test_scaled  = final_scaler.transform(X_test_pt)

# Simpan Scaler
joblib.dump(pt, os.path.join(dataset_path, "power_transformer.pkl"))
joblib.dump(final_scaler, os.path.join(dataset_path, "final_minmax_scaler.pkl"))
print("Stabilized Scaling Berhasil!")


--- STEP 2: STABILIZED YEO-JOHNSON PIPELINE ---
Stabilized Scaling Berhasil!


In [28]:
from sklearn.ensemble import IsolationForest

train_benign_scaled = X_train_scaled[y_train == 0]
iso_forest = IsolationForest(contamination=0.01, random_state=42, n_jobs=-1)
outlier_pred = iso_forest.fit_predict(train_benign_scaled)
train_benign_clean = train_benign_scaled[outlier_pred == 1]

print(f"Benign samples: {train_benign_scaled.shape[0]} -> {train_benign_clean.shape[0]} (IF Cleaned)")

joblib.dump(iso_forest, os.path.join(dataset_path, "iso_forest.pkl"))
np.save(os.path.join(dataset_path, "train_benign_clean.npy"), train_benign_clean)

Benign samples: 9373163 -> 9279431 (IF Cleaned)


In [29]:
dataset_split = {
    'X_train': X_train_scaled, 'y_train': y_train.values,
    'X_val': X_val_scaled, 'y_val': y_val.values,
    'X_test': X_test_scaled, 'y_test': y_test.values,
    'features': X_train.columns.tolist()
}
joblib.dump(dataset_split, os.path.join(dataset_path, "dataset_split.joblib"), compress=3)

with open(os.path.join(dataset_path, "selected_features.txt"), "w") as f:
    for feat in X_train.columns:
        f.write(feat + "\n")

print(f"\n--- RINGKASAN FINAL ---")
print(f"Jumlah Fitur Final: {len(X_train.columns)}")
print("Semua file berhasil disimpan di Drive.")


--- RINGKASAN FINAL ---
Jumlah Fitur Final: 48
Semua file berhasil disimpan di Drive.
